# Train CNN for Philippine Banknote Classification (skeleton)

This notebook contains a safe, high-level training pipeline using Keras. Replace dataset paths and expand augmentations when running locally.

In [25]:
import tensorflow as tf
from tensorflow.keras import layers
print('TensorFlow', tf.__version__)

# NOTE: This notebook is a scaffold. Provide your dataset under dataset/ folders and run training locally.

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
def build_model(input_shape=(128,128,1), num_classes=6):
    inputs = tf.keras.Input(shape=input_shape)
    x = layers.Conv2D(32,3,activation='relu')(inputs)
    x = layers.MaxPool2D()(x)
    x = layers.Conv2D(64,3,activation='relu')(x)
    x = layers.MaxPool2D()(x)
    x = layers.Flatten()(x)
    x = layers.Dense(128, activation='relu')(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    model = tf.keras.Model(inputs, outputs)
    return model

model = build_model()
model.summary()


NameError: name 'tf' is not defined

## Complete training pipeline

This section provides a minimal, end-to-end Keras training pipeline that expects a directory layout:

- `dataset/train/<class_name>/*.jpg`
- `dataset/val/<class_name>/*.jpg`

Adjust paths and parameters as needed. Run these cells before the TFLite conversion cells below.

In [ ]:
# Data pipeline
import os
import tensorflow as tf
from tensorflow.keras import layers

TRAIN_DIR = 'dataset/train'
VAL_DIR = 'dataset/val'
IMG_SIZE = (128, 128)
COLOR_MODE = 'grayscale'  # must match build_model input channels
BATCH_SIZE = 32
SEED = 1337

if not os.path.isdir(TRAIN_DIR) or not os.path.isdir(VAL_DIR):
    print('Dataset directories not found; please create dataset/train and dataset/val with class subfolders.')
else:
    train_ds = tf.keras.utils.image_dataset_from_directory(
        TRAIN_DIR,
        labels='inferred',
        label_mode='int',
        color_mode=COLOR_MODE,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=SEED,
    )
    val_ds = tf.keras.utils.image_dataset_from_directory(
        VAL_DIR,
        labels='inferred',
        label_mode='int',
        color_mode=COLOR_MODE,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        shuffle=False,
    )
    class_names = train_ds.class_names
    print('Classes:', class_names)

    norm = layers.Rescaling(1./255)
    def prep(ds, training=False):
        ds = ds.map(lambda x, y: (norm(x), y), num_parallel_calls=tf.data.AUTOTUNE)
        # TODO: add augmentations here if needed
        return ds.cache().prefetch(tf.data.AUTOTUNE)

    train_ds = prep(train_ds, True)
    val_ds = prep(val_ds, False)

ModuleNotFoundError: No module named 'tensorflow'

In [ ]:
# Compile, train, and save model
import math
from tensorflow.keras import optimizers

EPOCHS = 5  # adjust as needed
LR = 1e-3

if 'train_ds' in globals() and 'val_ds' in globals():
    num_classes = len(class_names)
    # Rebuild model if class count differs
    try:
        keras_model
    except NameError:
        keras_model = build_model(input_shape=(IMG_SIZE[0], IMG_SIZE[1], 1), num_classes=num_classes)

    keras_model.compile(
        optimizer=optimizers.Adam(LR),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )

    steps = None
    val_steps = None
    history = keras_model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        steps_per_epoch=steps,
        validation_steps=val_steps
    )

    # Save Keras model to disk
    keras_model.save('model.h5')
    print('Saved model.h5')
else:
    print('Datasets not available; skipping training. Prepare dataset directories and rerun.')

ModuleNotFoundError: No module named 'tensorflow'

## TFLite conversion — Model loading and preparation

This section converts the trained Keras model to TensorFlow Lite (TFLite), suitable for microcontroller deployment (e.g., TFLite Micro on ESP32).

Steps:
- Load an already-trained Keras model (from memory or `model.h5`).
- Configure conversion parameters (dynamic range or full int8 quantization).
- Execute the conversion and save `.tflite` file.
- Verify the converted model by running a test inference.

In [ ]:
# Model loading and preparation
import os
import numpy as np
import tensorflow as tf

MODEL_PATH = 'model.h5'

# Load trained model from disk if available, otherwise use the in-memory model
if os.path.exists(MODEL_PATH):
    print(f'Loading Keras model from {MODEL_PATH} ...')
    keras_model = tf.keras.models.load_model(MODEL_PATH)
else:
    print('No model.h5 found, using in-memory model (or building a fresh one).')
    try:
        keras_model = model
    except NameError:
        try:
            keras_model = build_model()
        except NameError:
            raise RuntimeError('No model instance found and build_model() is undefined in this session.')

keras_model.summary()

# Input shape inference (batch dimension will be added at inference time)
input_shape = tuple(keras_model.inputs[0].shape[1:])
print('Inferred input shape:', input_shape)

## Conversion parameters configuration

Configure quantization and (optional) representative dataset for full int8 quantization. If a representative dataset is not provided, dynamic range quantization will be used.

In [ ]:
# Conversion parameters configuration
USE_FULL_INT8 = True  # set False to use dynamic range quantization
REP_SAMPLES = 50

# Build a representative dataset generator for full-int8 quantization.
# Replace the random generator with real, preprocessed samples matching your training pipeline.
def representative_data_gen():
    for _ in range(REP_SAMPLES):
        # Assuming float32 input in [0,1] for the Keras model. Adjust if you used a different preprocessing.
        sample = np.random.rand(1, *input_shape).astype('float32')
        yield [sample]

converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

if USE_FULL_INT8:
    converter.representative_dataset = representative_data_gen
    converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    converter.inference_input_type = tf.uint8
    converter.inference_output_type = tf.uint8
    print('Configured converter for full int8 quantization.')
else:
    print('Configured converter for dynamic range quantization.')

## Conversion execution

Run the TFLite conversion and save the model to disk.

In [ ]:
# Conversion execution
TFLITE_PATH = 'model.tflite'

print('Converting to TFLite...')
tflite_model = converter.convert()

with open(TFLITE_PATH, 'wb') as f:
    f.write(tflite_model)

size_kb = len(tflite_model) / 1024
print(f'Saved {TFLITE_PATH} ({size_kb:.1f} KB)')

## Verification of converted model

Load the generated `.tflite` model, inspect I/O details, and run a test inference to verify successful conversion.

In [ ]:
# Verification of converted model
import os
import numpy as np
import tensorflow as tf

if not os.path.exists(TFLITE_PATH):
    raise FileNotFoundError(f'{TFLITE_PATH} not found. Run the conversion cell first.')

interpreter = tf.lite.Interpreter(model_path=TFLITE_PATH)
interpreter.allocate_tensors()
input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print('Input details:', input_details)
print('Output details:', output_details)

# Prepare a dummy input matching the model's expected dtype and shape
in_shape = input_details[0]['shape']
indtype = input_details[0]['dtype']

sample = np.random.rand(*in_shape).astype('float32')

if indtype == np.uint8:
    scale, zero = input_details[0]['quantization']
    if scale == 0:
        input_data = np.clip(sample, 0, 255).astype('uint8')
    else:
        input_data = np.clip(np.round(sample / scale + zero), 0, 255).astype('uint8')
else:
    input_data = sample.astype(indtype)

interpreter.set_tensor(input_details[0]['index'], input_data)
interpreter.invoke()

output = interpreter.get_tensor(output_details[0]['index'])

# Dequantize for readability if needed
if output_details[0]['dtype'] == np.uint8:
    oscale, ozero = output_details[0]['quantization']
    if oscale != 0:
        output_readable = (output.astype('float32') - ozero) * oscale
    else:
        output_readable = output.astype('float32')
else:
    output_readable = output

print('Output vector (first item):')
print(output_readable[0])
print('Sum:', float(np.sum(output_readable[0])))